# Synthetic white-light polarised brightness (pB) — three test datasets

Thomson-scattering pB via `rrt.wl.run_siddon_pB` (van de Hulst / Billings kernel), with the optional NRGF radial filter. White light depends on **density only** — no temperature field and no AIA response are needed.

Each test case is a YAML config in `examples/configs/` — the notebook loads the *same* file the CLI does, so `rrt-wl <config>` and this notebook always render the same thing.

| `CASE` | data | config | notes |
|---|---|---|---|
| `case1` | `rho_bin.h5` | [`case1_wl.yaml`](../configs/case1_wl.yaml) | MAS/PSI shell, code units, sub-Earth at 2024-03-30 20:49 UTC, FOV 6 R☉ |
| `case2` | `mhd_data_0220.h5` | [`case2_wl.yaml`](../configs/case2_wl.yaml) | ARMS 90° wedge, density already in cm⁻³, FOV 5 R☉ |
| `case3` | `rhoC_02.h5` | [`case3_wl.yaml`](../configs/case3_wl.yaml) | MAS/PSI shell, different snapshot, FOV 6 R☉ |

Raw pB spans many decades and looks like a featureless glow; the **NRGF** panel `(pixel − μ_r)/σ_r` per radial annulus is what makes streamers and CME fronts visible.

In [ ]:
import os
from pathlib import Path

import numpy as np, matplotlib.pyplot as plt
from rrt.config import load_config, load_cube_from_config
from rrt.observer import image_grid, make_observer, view_angles
from rrt.wl import run_siddon_pB
from rrt.viz import nrgf, plot_map

# configs use repo-relative paths (data/…), so work from the repo root
REPO = Path.cwd().resolve()
while not (REPO / 'examples' / 'configs').exists() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)
print('working from:', REPO)

## 1. Pick the test case

Change `CASE` and re-run everything below. `r_range` matters more here than for EUV: pB is a coronagraph product, so keep the outer corona (the MAS cases integrate out to 15 R☉ even though the image stops at 6).

In [ ]:
CASE = 'case1'          # 'case1' | 'case2' | 'case3'

CONFIGS = {
    'case1':    'examples/configs/case1_wl.yaml',
    'case2': 'examples/configs/case2_wl.yaml',
    'case3':    'examples/configs/case3_wl.yaml',
}

cfg = load_config(CONFIGS[CASE])
cube = load_cube_from_config(cfg)
print(cube.summary())
r, th, ph, ne = cube.r, cube.theta, cube.phi, cube.ne

VIEWS = {v['label']: make_observer(*view_angles(v)) for v in cfg.views}
OBS_TIME = next((v.get('obs_time') for v in cfg.views if v.get('obs_time')), '')
print(f'{CASE}: {cfg.Npix}² px, Rmax = {cfg.Rmax} R_sun, Rocc = {cfg.Rocc} R_sun, '
      f'u = {cfg.U_LIMB}, views = {list(VIEWS)}')

## 2. Integrate pB per viewpoint

`pB(pixel) = ∫ n_e(r)·K(r) dℓ` with `K(r) = σ_e·[(1−u)·A(r) + u·B(r)]`. `Rocc` blanks the pixels behind the occulter. The NRGF is computed on the same grid right after.

In [ ]:
Xg, Yg, _ = image_grid(cfg.Rmax, cfg.Npix)
pb, pb_nrgf = {}, {}
for label, (n_obs, e_los, x_img, y_img) in VIEWS.items():
    print(f'--- {label} ---')
    pb[label] = run_siddon_pB(Xg, Yg, x_img, y_img, e_los, r, th, ph, ne,
                              Npix=cfg.Npix, Rmax=cfg.Rmax, Rocc=cfg.Rocc,
                              U_LIMB=cfg.U_LIMB, SIGMA_T=cfg.SIGMA_T,
                              plot=False, save=False)
    pb_nrgf[label] = nrgf(pb[label], Xg, Yg, r_inner=cfg.Rocc, r_outer=cfg.Rmax)

## 3. Raw pB vs NRGF

One row per viewpoint: raw pB on a log scale (left) and the NRGF in units of σ per annulus (right). Files are written to the config's `output_dir` with an `nb_` prefix, so they never overwrite what `rrt-wl` writes.

In [ ]:
OUT_DIR = Path(cfg.output_dir); OUT_DIR.mkdir(parents=True, exist_ok=True)
slug = lambda s: ''.join(c if c.isalnum() else '_' for c in s).strip('_').lower()

nrow = len(VIEWS)
fig, axes = plt.subplots(nrow, 2, figsize=(11, 5.2 * nrow), squeeze=False)
for row, label in enumerate(VIEWS):
    plot_map(pb[label], cfg.Rmax, cmap='Oranges_r', log=True, occulter=cfg.Rocc,
             cbar_label='pB', title=f'{label} — raw pB', ax=axes[row, 0])
    plot_map(pb_nrgf[label], cfg.Rmax, cmap='Oranges_r', log=False, occulter=cfg.Rocc,
             vlim=(-2, 3), cbar_label=r'NRGF [$\sigma$]',
             title=f'{label} — NRGF', ax=axes[row, 1])
    for tag, img, is_log in (('pb', pb[label], True), ('nrgf', pb_nrgf[label], False)):
        f1, _, _ = plot_map(img, cfg.Rmax, cmap='Oranges_r', log=is_log, occulter=cfg.Rocc,
                            vlim=None if is_log else (-2, 3),
                            cbar_label=('pB' if is_log else r'NRGF [$\sigma$]'),
                            title=f'{CASE} · {label} — white light ({tag})  {OBS_TIME}')
        f1.savefig(OUT_DIR / f'nb_wl_{tag}_{slug(label)}.png', dpi=150, bbox_inches='tight')
        plt.close(f1)

fig.suptitle(f'{CASE} — Thomson-scattering pB   {OBS_TIME}', y=1.0)
fig.tight_layout()
fig.savefig(OUT_DIR / 'nb_wl_all_views.png', dpi=150, bbox_inches='tight')
print('wrote →', OUT_DIR)
plt.show()